In [2]:
!pip install -qU langchain-openai
!pip install langchain-community


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 425.3 kB/s  0:00:08 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 179.9 kB/s  0:00:08 eta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 267.1 kB/s  0:00:07 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16/16 [langchain-community]ngchain-community]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [1]:
import fitz  # PyMuPDF
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# 1. Load PDF
doc = fitz.open("2025舞光LED21st(單頁水印可搜尋).pdf")
text = "".join([page.get_text() for page in doc])

# 2. Chunking
# We split text so it fits in the LLM's context window
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
chunks = text_splitter.split_text(text)

# 3. Embedding & Storage
# This converts text into a list of numbers (vectors)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma.from_texts(chunks, embeddings, persist_directory="./chroma_db")

/Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# 1. Setup Retriever
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 2. Define the LLM
llm = ChatOpenAI(model_name="gpt-5.2", temperature=0)

# 3. Create the Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# 4. Build the Chain using LCEL (Modern 1.2.x style)
# This replaces RetrievalQA entirely
qa_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 5. Ask a question
# Note: Since it's a simple LCEL chain, you pass the string directly
response = qa_chain.invoke("我要找放在浴室的燈?")
print(response)

從提供的型錄內容來看，沒有出現「浴室燈」或明確標示可用於浴室（例如防水等級 IP44／IP65、可用於潮濕室內）的室內燈具頁面。

目前與「可能靠近浴室需求」較接近的只有：
- **戶外照明的防水壁燈**（例如：**艾莎防水壁燈 IP44**、瑞絲防水壁燈等）──但型錄描述用途是「建築外牆、門廊等空間」，並未寫可用於浴室。
- **感應器**（室內微波/紅外線感應器）──它是控制用配件，不是燈具本體，也未標示浴室適用。

如果你要我用這份資料幫你「挑得到」浴室可用的燈，請你再補一頁/更完整的浴室燈或室內吸頂燈（含防水等級）的型錄內容；或告訴我你浴室想裝的是**吸頂燈/壁燈/鏡前燈**以及是否需要**防水 IP 等級**。


In [7]:
import sys
import langchain
print(f"Python Path: {sys.executable}")
print(f"LangChain Version: {langchain.__version__}")
print(f"LangChain Location: {langchain.__file__}")

Python Path: /Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/bin/python
LangChain Version: 1.2.10
LangChain Location: /Users/max/Desktop/python_codes/DanceLight--LED-AI-Chatbot-and-attribute-searching_v2/.venv/lib/python3.11/site-packages/langchain/__init__.py
